# L18 · 可逆性与秩 — 秩 = 信息量，零空间 = 被压缩的方向，奇异矩阵诊断

**目标**：判断一个方阵是否可逆的三条判据，以及「严格对角占优」这一快速充分条件。

**为什么对 Aurora 重要**：Jacobi/Gauss-Seidel 迭代求解器以对角占优为收敛前提；`is_sdd` 检查是求解器入口的第一道关卡。

## 本课剧情：判断过程能不能倒带

可逆性问一个朴素问题：看到输出以后，我们还能不能找回唯一输入？`det(A) ≠ 0` 和「所有特征值非零」是两条充要条件，严格对角占优是不用算特征值就能快速确认可逆的充分条件。

## 1. 三条判据

1. **行列式 ≠ 0**（充要）
2. **所有特征值 ≠ 0**（充要）
3. **严格对角占优 (s.d.d.)**：每行对角元的绝对值 > 该行其余元素绝对值之和（**充分**但非必要）。

三条判据的实用定位不同：`det ≠ 0` 和「所有特征值 ≠ 0」是充要条件，两者等价，但计算代价都是 O(n³)（LU 分解 / QR 迭代）。严格对角占优只是充分条件——不满足它矩阵仍可能可逆——但检查代价仅 O(n²)，且它直接保证 Jacobi/Gauss-Seidel 迭代的谱半径 < 1，即迭代误差必然收敛。实际工程策略：先用 `is_sdd` 快速筛查；通过则启动迭代求解器；不通过再按需计算 det 或特征值。

## 符号入口：先看形状，再看运算

本节处理方阵 `A`（shape `(n, n)`），核心工具是 `np.linalg.det(A)` 和 `np.linalg.eigvals(A)`。判断可逆性归结为两个数字：一个行列式标量和一组特征值向量。

In [ ]:
import numpy as np
A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
print('det =', round(np.linalg.det(A)))
print('特征值:', np.round(np.linalg.eigvals(A), 3))
print('两条充要判据都说：可逆 =', abs(np.linalg.det(A))>1e-9)

## 动手观察：从 det 和特征值读可逆性

运行下面的代码，注意 `det` 接近零时特征值中是否出现接近零的分量——两条充要判据在数值上指向同一个事实。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：观察矩阵如何作用于多个向量

可逆矩阵把不同的输入映射到不同的输出；奇异矩阵则会把两个不同向量压缩到同一个方向。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. ✏️ 实现 `is_sdd(A)`（严格对角占优）

对每行 i：`|A[i,i]| > Σ_{j≠i} |A[i,j]|` 是否都成立。

**推理路线**：
1. 取第 i 行的对角元绝对值：`d = abs(A[i, i])`。
2. 取第 i 行所有元素绝对值之和：`s = sum(abs(A[i]))`；非对角部分之和为 `s - d`（无需单独遍历 j≠i）。
3. 检查 `d > s - d` 对所有行成立；用 `all(...)` 把 n 个布尔值合并为一个返回值。

**参考输入输出**：`A=[[4,1],[1,3]]` → 行0：4>1 ✓；行1：3>1 ✓ → `True`；`B=[[1,2],[2,1]]` → 行0：1<2 ✗ → `False`

<details><summary>点击查看参考实现</summary>

```python
def is_sdd(A):
    A = np.asarray(A, float)
    n = len(A)
    return all(abs(A[i, i]) > np.sum(np.abs(A[i])) - abs(A[i, i]) for i in range(n))

# 向量化等价写法：
# diag = np.abs(np.diag(A))
# off  = np.sum(np.abs(A), axis=1) - diag
# return bool(np.all(diag > off))
```

</details>

### 写 `is_sdd` 前明确三件事

- 输入：`A`，shape `(n, n)` 的方阵
- 关键步骤：对每行 `i`，比较 `abs(A[i,i])` 与 `sum(abs(A[i])) - abs(A[i,i])`
- 返回：`bool`，所有行均满足则为 `True`

In [ ]:
def is_sdd(A):
    A = np.asarray(A, float)
    # ✏️ TODO: 返回是否严格对角占优 (True/False)
    return None


In [ ]:
# 课件三例 (page 171)
M1 = np.array([[2,0,1],[1,4,2],[1,3,6]], float)   # s.d.d. 且可逆
M2 = np.array([[1,0,2],[2,5,1],[3,2,13]], float)  # 非 s.d.d. 但仍可逆
M3 = np.array([[1,1],[1,1]], float)               # 非 s.d.d. 且不可逆
for name, M in [('M1',M1),('M2',M2),('M3',M3)]:
    print(f'{name}: sdd={is_sdd(M)}, 可逆={abs(np.linalg.det(M))>1e-9}')
assert is_sdd(M1) and not is_sdd(M2) and not is_sdd(M3)
print('\n✅ 与课件三例一致。注意 M2 说明：sdd 是充分非必要条件。')

**🔗 Aurora 连接**：`is_sdd` 是 Aurora 迭代求解器的入口卫士——系数矩阵只有通过 s.d.d. 检查，Jacobi/Gauss-Seidel 求解流程才会启动；此时谱半径 < 1 有数学保证，迭代误差按几何级数衰减而非发散。

**补充例题对应**：三条可逆性判据 + 严格对角占优。

**线代前导章节完成**：p1–p10 覆盖向量、矩阵、方程组、行列式、特征值与可逆性。

## 🎨 图示：严格对角占优矩阵(对角线显著占优 ⇒ 可逆)

In [ ]:
from laviz import style, heatmap
style()
heatmap([[2,0,1],[1,4,2],[1,3,6]], title='严格对角占优 ⇒ 可逆', cmap='magma');

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：Jacobi 迭代的收敛与发散

构造一个对角元明显大于同行其余元素绝对值之和的矩阵（如 `A = [[10,1,1],[1,10,1],[1,1,10]]`），用 Jacobi 迭代求解线性系统 `Ax = b`，打印每轮残差 `‖Ax_k - b‖`，确认误差逐步缩小至收敛。

再把对角元缩小使矩阵不再满足 s.d.d.（如 `A = [[1,2,2],[2,1,2],[2,2,1]]`），重跑相同迭代步数，观察残差不减反增或振荡。两次实验对比说明：`is_sdd` 通过是 Jacobi 收敛的理论保证，不是可有可无的布尔标记。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

现在能用 `is_sdd(A)` 逐行检查严格对角占优，用 `np.linalg.det` 和 `np.linalg.eigvals` 验证两条充要判据。`is_sdd` 对应 Aurora 迭代求解器的入口检查：通过它，Jacobi 迭代才能保证收敛。下一节 SVD 把可逆性推广到矩形矩阵：奇异值替代特征值，成为衡量矩阵「可逆程度」的连续尺度。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
